#### Simple Gen AI APP Using Langchain

In [98]:
import os
from dotenv import load_dotenv
load_dotenv()
# os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
# ## Langsmith Tracking
# os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_TRACING_V2"]="true"
# os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

True

In [99]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

In [100]:

loader=WebBaseLoader("https://arxiv.org/pdf/1706.03762.pdf")
loader

In [101]:
from langchain_docling import DoclingLoader

FILE_PATH = "https://arxiv.org/pdf/1706.03762.pdf"

loader = DoclingLoader(file_path=FILE_PATH)

In [102]:
docs = loader.load()
docs

[INFO] 2026-05-21 17:48:56,109 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-21 17:48:56,114 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-21 17:48:56,161 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\soura\Desktop\AgenticAI\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-21 17:48:56,162 [RapidOCR] main.py:50: Using C:\Users\soura\Desktop\AgenticAI\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-21 17:48:56,405 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-21 17:48:56,406 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-05-21 17:48:56,409 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\soura\Desktop\AgenticAI\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-21 17:48:56,410 [RapidOCR] main.py:50: Using C:\Users\soura\Desktop\AgenticAI\.venv\Lib\site-packages\rapidocr\models\ch

[Document(metadata={'source': 'https://arxiv.org/pdf/1706.03762.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 173]}]}], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 2949302674760005271, 'filename': '1706.03762.pdf'}}}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.'),
 Document(metadata={'source': 'https://arxiv.org/pdf/1706.03762.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children'

In [103]:

### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [104]:
documents

[Document(metadata={'source': 'https://arxiv.org/pdf/1706.03762.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 124.313, 't': 717.8198352, 'r': 487.89454240000015, 'b': 679.662512679646, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 173]}]}], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 2949302674760005271, 'filename': '1706.03762.pdf'}}}, page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.'),
 Document(metadata={'source': 'https://arxiv.org/pdf/1706.03762.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children'

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [106]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load free local embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7367.97it/s]


In [107]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings1 = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4771.76it/s]


In [108]:

from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [109]:
from langchain_community.vectorstores import FAISS
vectorstoredb1=FAISS.from_documents(documents,embeddings1)

In [110]:
vectorstoredb


In [111]:
vectorstoredb1


In [112]:
## Query From a vector db
query="What is attention mechanism?"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum'

In [113]:
import langchain; import langchain_community
print(langchain.__version__, langchain_community.__version__)

1.3.0 0.4.1


In [ ]:

# from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents.stuff import create_stuff_documents_chain
# from langchain_openai import ChatOpenAI  # Use your chosen LLM wrapper

# llm = ChatOpenAI()   # Or other supported LLM

# prompt = ChatPromptTemplate.from_template("""
# Answer the following question based only on the provided context:
# <context>
# {context}
# </context>
# Question: {input}
# """)

# document_chain = create_stuff_documents_chain(
#     llm=llm,
#     prompt=prompt,
#     document_separator="\n\n",      # Optional: customize separator
#     document_variable_name="context" # match your prompt field
# )

# # Usage example (requires retriever integration for full RAG)
# result = document_chain.invoke({
#     "input": "YOUR QUESTION HERE",
#     "context": [your_documents],    # List of Document objects
# })

# print(result)

In [116]:

from langchain_groq import ChatGroq
llm=ChatGroq(model="openai/gpt-oss-120b")
print(llm)

output_version=None profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x000001E18D160550> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E18D160F50> model_name='openai/gpt-oss-120b' model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


In [115]:

# ## Retrieval Chain, Document chain

# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

# document_chain=create_stuff_documents_chain(llm,prompt)
# document_chain

In [117]:
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()
document_chain=prompt|llm|output_parser


In [118]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'LangSmith’s usage is limited by two metrics:\n\n1. **Total traces** – the overall number of traces you can record.  \n2. **Extended traces** – a separate cap for traces that use extended (more detailed) logging.\n\nThese two limits correspond to the two metrics shown on the LangSmith usage graph.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [119]:

### Input--->Retriever--->vectorstoredb

vectorstoredb1

In [87]:
# retriever=vectorstoredb1.as_retriever()
# from langchain.chains import create_retrieval_chain
# retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [88]:

# retrieval_chain

In [89]:

## Get the response form the LLM
# response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
# response['answer']

# response=document_chain.invoke({"input":"Can you tell me about Prathamesh in 20 words?"})
# print(response)

In [120]:
# ✅ LangChain v1.0 — FAISS + as_retriever() + Runnable RAG (working)

from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ✅ 1) Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ✅ 2) Create FAISS vector store
vectorstore = FAISS.from_documents(documents, embeddings)

# ✅ 3) Convert to retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# ✅ 4) Prompt
prompt = ChatPromptTemplate.from_template("""
Answer using ONLY the following context:

<context>
{context}
</context>

Question: {input}
""")

# ✅ 5) Correct Runnable RAG pipeline
rag_chain = (
    {
        "context": itemgetter("input") | retriever,
        "input": itemgetter("input")
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ✅ 6) Run
response = rag_chain.invoke({"input": "What is self attention?"})

print("\r\nANSWER:\r\n", response)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6952.71it/s]



ANSWER:
 Self‑attention, sometimes called **intra‑attention**, is an attention mechanism that relates different positions of a single sequence in order to compute a representation of that sequence.


In [121]:

response = rag_chain.invoke({"input": "Explain transformer architecture"})

print("\r\nANSWER:\r\n", response)


ANSWER:
 The Transformer is built around the classic **encoder‑decoder** framework for sequence‑to‑sequence learning.  

* **Encoder** – The encoder receives an input sequence of symbol representations \((x_1,\dots ,x_n)\) and converts it into a sequence of continuous vectors \(\mathbf{z} = (z_1,\dots ,z_n)\).  This conversion is performed by **stacked layers** that consist of **self‑attention** mechanisms followed by **point‑wise, fully‑connected feed‑forward** networks.  

* **Decoder** – Using the encoded representations \(\mathbf{z}\), the decoder generates the output sequence \((y_1,\dots ,y_m)\) one symbol at a time.  Like the encoder, each decoder layer contains self‑attention and point‑wise fully‑connected components, but the decoder is **auto‑regressive**: at each step it also takes as input the symbols that have already been generated.  

Thus, the overall architecture is a pair of parallel stacks (the left half of Figure 1 for the encoder, the right half for the decoder) th

In [122]:
response = rag_chain.invoke({"input": "Why are transformers better than RNNs"})

print("\r\nANSWER:\r\n", response)


ANSWER:
 Transformers improve on recurrent neural networks because they discard the sequence‑aligned recurrence that RNNs require and instead compute every representation with **self‑attention**.  
- The Transformer “relies entirely on self‑attention to compute representations of its input and output without using sequence‑aligned RNNs or convolution” (Background).  
- By using stacked self‑attention (and simple point‑wise feed‑forward layers) in both the encoder and decoder, the model can process all positions in parallel rather than step‑by‑step, which gives it clear advantages over earlier RNN‑based transduction models.  

Thus, transformers are considered better than RNNs because they replace recurrent processing with self‑attention, enabling more efficient and powerful sequence‑to‑sequence modeling.
